# Seasonal Agriculture Performance Analysis

## VOIS / AICTE Data Analytics Major Project

This notebook analyzes the **Seasonal Agriculture Performance dataset** using Python and common data-analytics techniques.

### Project objectives
- Understand the structure and quality of the agricultural dataset.
- Clean missing and duplicate records.
- Compare agricultural performance across **Kharif, Rabi, and Zaid** seasons.
- Analyze crop yield, production, revenue, cost, profit, and water efficiency.
- Compare irrigation methods.
- Study relationships between environmental/resource variables and yield.
- Generate visualizations and derive actionable insights.

> **Note:** Personal details are intentionally not included in this notebook.

## 1. Import Libraries

The analysis uses:
- **Pandas** for data loading, cleaning, grouping, and analysis.
- **NumPy** for numerical operations.
- **Matplotlib** for visualizations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load the Dataset

Place `seasonal_agriculture_performance_dataset.csv` in the same folder as this notebook before running it.

In [ ]:
DATA_FILE = "seasonal_agriculture_performance_dataset.csv"
df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()

## 3. Dataset Overview

The dataset contains **4,000 records and 28 columns** covering farm location, crop and season, farm area, environmental conditions, soil and input usage, irrigation, yield, production, market price, cost, revenue, profit, water usage, water efficiency, and disease/pest risk.

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nBasic information:")
df.info()

## 4. Data Quality Check

We check:
1. Missing values
2. Duplicate rows
3. Unique values in important categorical columns
4. Descriptive statistics

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print("Missing values by column:")
display(missing[missing > 0])

print("Total duplicate rows:", df.duplicated().sum())

for col in ["State", "District", "Crop", "Season", "Irrigation_Method"]:
    print(f"\nUnique values in {col}:")
    print(sorted(df[col].dropna().astype(str).unique()))

In [ ]:
print("Descriptive statistics for numeric variables:")
display(df.describe().T)

## 5. Data Cleaning

The dataset contains missing values in some numeric fields. For this analysis, missing numeric observations are handled using **median imputation**, which is less sensitive to extreme values than mean imputation.

Duplicate rows are checked and removed if present.

Categorical missing values, if any, are filled with `"Unknown"`.

In [ ]:
df_clean = df.copy()

# Remove exact duplicate records
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# Numeric columns: median imputation
numeric_cols = df_clean.select_dtypes(include=np.number).columns
for col in numeric_cols:
    if df_clean[col].isna().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Categorical columns: explicit placeholder
categorical_cols = df_clean.select_dtypes(exclude=np.number).columns
for col in categorical_cols:
    if df_clean[col].isna().any():
        df_clean[col] = df_clean[col].fillna("Unknown")

print("Rows after duplicate removal:", len(df_clean))
print("Remaining missing values:", int(df_clean.isna().sum().sum()))

### Data-quality summary

The original dataset has **4,000 rows and 28 columns**. The observed missing values include:
- Rainfall: **48**
- Soil moisture: **40**
- Yield: **32**

There are **no duplicate rows** in the supplied dataset. After the cleaning step, the analysis dataset contains no missing values.

## 6. Univariate Analysis

This section examines the distribution of major performance variables such as yield, rainfall, total cost, revenue, profit, and water efficiency.

In [ ]:
key_numeric = [
    "Yield_Tonnes_Ha",
    "Rainfall_mm",
    "Total_Cost_INR",
    "Revenue_INR",
    "Profit_INR",
    "Water_Efficiency_t_per_1000m3"
]

display(df_clean[key_numeric].describe().T)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df_clean["Yield_Tonnes_Ha"], bins=30)
ax.set_title("Distribution of Crop Yield")
ax.set_xlabel("Yield (Tonnes/Ha)")
ax.set_ylabel("Number of Farms")
plt.tight_layout()
plt.show()

## 7. Seasonal Performance Comparison

The main objective is to compare **Kharif, Rabi, and Zaid** using:
- Average yield
- Average production
- Average revenue
- Average cost
- Average profit
- Average water use
- Average water efficiency

In [ ]:
season_summary = (
    df_clean.groupby("Season")
    .agg(
        Farms=("Farm_ID", "count"),
        Avg_Yield_Tonnes_Ha=("Yield_Tonnes_Ha", "mean"),
        Avg_Production_Tonnes=("Production_Tonnes", "mean"),
        Avg_Revenue_INR=("Revenue_INR", "mean"),
        Avg_Cost_INR=("Total_Cost_INR", "mean"),
        Avg_Profit_INR=("Profit_INR", "mean"),
        Avg_Water_Used_m3=("Water_Used_m3", "mean"),
        Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean")
    )
    .sort_values("Avg_Yield_Tonnes_Ha", ascending=False)
)

display(season_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
season_summary["Avg_Yield_Tonnes_Ha"].plot(kind="bar", ax=ax)
ax.set_title("Average Yield by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Average Yield (Tonnes/Ha)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
season_summary["Avg_Profit_INR"].plot(kind="bar", ax=ax)
ax.set_title("Average Profit by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Average Profit (INR)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

### Seasonal findings

From the supplied dataset:
- **Kharif** has the highest average yield at approximately **5.64 tonnes/ha**.
- **Rabi** follows with approximately **5.08 tonnes/ha**.
- **Zaid** has the lowest average yield at approximately **4.67 tonnes/ha**.
- Kharif also has the highest average profit, approximately **₹1.79 lakh per farm**.
- Zaid has the weakest economic performance and an average loss of approximately **₹24,805 per farm**.

## 8. Crop-Level Analysis

Crop performance is compared using average yield, revenue, cost, and profit.

In [ ]:
crop_summary = (
    df_clean.groupby("Crop")
    .agg(
        Farms=("Farm_ID", "count"),
        Avg_Yield_Tonnes_Ha=("Yield_Tonnes_Ha", "mean"),
        Avg_Revenue_INR=("Revenue_INR", "mean"),
        Avg_Cost_INR=("Total_Cost_INR", "mean"),
        Avg_Profit_INR=("Profit_INR", "mean")
    )
    .sort_values("Avg_Profit_INR", ascending=False)
)

display(crop_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
crop_summary["Avg_Profit_INR"].plot(kind="bar", ax=ax)
ax.set_title("Average Profit by Crop")
ax.set_xlabel("Crop")
ax.set_ylabel("Average Profit (INR)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### Crop findings

The supplied data indicates:
- **Sugarcane** has the highest average yield, approximately **46.94 tonnes/ha**, and the highest average profit, approximately **₹8.17 lakh per farm**.
- **Chilli** has the second-highest average profit, approximately **₹7.51 lakh per farm**.
- Wheat, Rice, and Maize show negative average profit in this dataset.

These results describe the supplied dataset and should not automatically be generalized to all farms or regions.

## 9. Irrigation Method Analysis

Irrigation is evaluated in terms of yield, profit, water use, and water efficiency.

In [ ]:
irrigation_summary = (
    df_clean.groupby("Irrigation_Method")
    .agg(
        Farms=("Farm_ID", "count"),
        Avg_Yield_Tonnes_Ha=("Yield_Tonnes_Ha", "mean"),
        Avg_Profit_INR=("Profit_INR", "mean"),
        Avg_Water_Used_m3=("Water_Used_m3", "mean"),
        Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean")
    )
    .sort_values("Avg_Yield_Tonnes_Ha", ascending=False)
)

display(irrigation_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
irrigation_summary["Avg_Yield_Tonnes_Ha"].plot(kind="bar", ax=ax)
ax.set_title("Average Yield by Irrigation Method")
ax.set_xlabel("Irrigation Method")
ax.set_ylabel("Average Yield (Tonnes/Ha)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

### Irrigation findings

- **Drip irrigation** shows the highest average yield, approximately **6.62 tonnes/ha**, and the highest average profit, approximately **₹2.20 lakh per farm**.
- **Rainfed** farming shows the highest average water-efficiency value, approximately **7.56 tonnes per 1,000 m³**.
- **Flood irrigation** has the highest average water use, approximately **8,026 m³**, and the lowest water-efficiency value, approximately **3.44 tonnes per 1,000 m³**.

## 10. Economic Analysis

Revenue, cost, and profit are compared across seasons.

**Profit is interpreted as: Revenue − Total Cost.**

In [ ]:
economic_by_season = (
    df_clean.groupby("Season")
    .agg(
        Avg_Revenue_INR=("Revenue_INR", "mean"),
        Avg_Cost_INR=("Total_Cost_INR", "mean"),
        Avg_Profit_INR=("Profit_INR", "mean")
    )
)

display(economic_by_season)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
economic_by_season.plot(kind="bar", ax=ax)
ax.set_title("Seasonal Economic Performance")
ax.set_xlabel("Season")
ax.set_ylabel("Average INR per Farm")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

### Economic interpretation

Kharif combines the strongest average revenue and profit performance in the dataset. Zaid has the lowest average economic outcome, with average profit below zero. This suggests that season selection and the relationship between crop choice, costs, market price, and expected yield are important economic considerations.

## 11. Environmental Factors and Yield

The project also examines whether rainfall, temperature, fertilizer use, and other numeric factors are associated with yield.

Correlation measures **linear association**; it does not prove that one variable causes another.

In [ ]:
correlation_cols = [
    "Yield_Tonnes_Ha",
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Humidity_pct",
    "Sunlight_Hours_Day",
    "Soil_pH",
    "Soil_Moisture_pct",
    "Nitrogen_kg_ha",
    "Phosphorus_kg_ha",
    "Potassium_kg_ha",
    "Fertilizer_kg_ha",
    "Pesticide_Litre_ha",
    "Seed_Quality_Score",
    "Water_Used_m3",
    "Water_Efficiency_t_per_1000m3",
    "Disease_Pest_Risk_pct",
    "Production_Tonnes",
    "Profit_INR"
]

corr = df_clean[correlation_cols].corr(numeric_only=True)
display(corr["Yield_Tonnes_Ha"].sort_values(ascending=False).to_frame("Correlation with Yield"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticklabels(corr.index, fontsize=7)
ax.set_title("Correlation Matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

### Important correlation results

For the supplied dataset:
- Yield vs. Water Efficiency: **r ≈ 0.915** — very strong positive association.
- Yield vs. Production: **r ≈ 0.885** — strong positive association.
- Yield vs. Profit: **r ≈ 0.490** — moderate positive association.
- Rainfall vs. Yield: **r ≈ 0.031** — very weak linear association.
- Fertilizer vs. Yield: **r ≈ 0.001** — essentially no linear association in this dataset.
- Temperature vs. Yield: **r ≈ 0.009** — very weak linear association.

A correlation value alone should not be interpreted as proof of causation.

## 12. Rainfall vs Yield

A scatter plot is used to visually inspect the relationship between rainfall and yield.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_clean["Rainfall_mm"], df_clean["Yield_Tonnes_Ha"], alpha=0.45)
ax.set_title("Rainfall vs Crop Yield")
ax.set_xlabel("Rainfall (mm)")
ax.set_ylabel("Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

The dataset shows a very weak linear relationship between rainfall and yield (approximately **r = 0.031**). This means rainfall alone does not explain much of the variation in yield in the supplied records. Other factors such as crop type, irrigation, soil conditions, inputs, and management practices may also matter.

## 13. Water Efficiency vs Yield

Water efficiency is compared with yield because efficient use of water is an important resource-performance measure.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    df_clean["Water_Efficiency_t_per_1000m3"],
    df_clean["Yield_Tonnes_Ha"],
    alpha=0.45
)
ax.set_title("Water Efficiency vs Crop Yield")
ax.set_xlabel("Water Efficiency (Tonnes / 1,000 m³)")
ax.set_ylabel("Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

The supplied data shows a **very strong positive correlation (r ≈ 0.915)** between water efficiency and yield. This is an important analytical finding, although it should still be interpreted as an association rather than proof that improving water efficiency alone will cause higher yield.

## 14. Key Findings and Recommendations

### Key findings
1. **Kharif** has the highest average yield and strongest average profit among the three seasons.
2. **Zaid** has the lowest average yield and negative average profit in the supplied records.
3. **Drip irrigation** has the strongest average yield and profit among the irrigation methods.
4. **Rainfed** farming has the highest average water-efficiency value.
5. **Flood irrigation** uses the most water and has the lowest water-efficiency value.
6. **Sugarcane** has the highest average yield and profit among the crops in this dataset.
7. Yield has a very strong positive association with water efficiency and production.
8. Rainfall, temperature, and fertilizer show very weak linear correlations with yield in these records.

### Recommendations based on the dataset
- Consider **efficient irrigation methods**, especially drip, where practical and economically feasible.
- Monitor water use and prioritize practices that improve **water efficiency**.
- Evaluate **crop and season combinations** using expected yield, cost, market price, and profit rather than yield alone.
- Investigate the causes of poor economic performance during **Zaid**.
- Use additional years and field-level context before making broad agricultural policy decisions.

## 15. Conclusion

The Seasonal Agriculture Performance Analysis demonstrates how data analytics can be used to compare agricultural outcomes across seasons, crops, and irrigation methods.

The analysis identifies **Kharif as the strongest overall season in the supplied dataset**, while **Zaid shows weaker economic performance**. Irrigation analysis indicates that **drip irrigation is associated with higher average yield and profit**, while rainfed farming has the highest water-efficiency value. The correlation analysis highlights a strong association between yield and water efficiency, whereas rainfall and fertilizer have very weak linear relationships with yield in these records.

Overall, the project shows how **data cleaning, descriptive statistics, grouping, correlation analysis, and visualization** can convert raw agricultural records into understandable findings that can support better resource and farm-management decisions.

## 16. Future Scope

The project can be extended through:
- Multi-year agricultural data for trend analysis.
- Predictive models for yield and profit forecasting.
- Machine-learning models for crop/season recommendations.
- Geospatial analysis using farm locations.
- Power BI or Tableau dashboards for interactive reporting.
- Crop-specific resource optimization.
- Field validation with agricultural experts and real-world farm observations.

## 17. Reproducibility

To run this notebook:

1. Install Python and Jupyter Notebook/JupyterLab.
2. Install the required packages:
   `pandas`, `numpy`, `matplotlib`
3. Keep this notebook and `seasonal_agriculture_performance_dataset.csv` in the same folder.
4. Run the notebook from the first cell to the last cell.

**File used:** `seasonal_agriculture_performance_dataset.csv`